# 2D-ALiBi Training on ImageNet-100

This notebook trains **2D-ALiBi** ViT-Base models for the TPAMI revision.

**What it does**:
1. Mounts Google Drive
2. Loads your existing `full_scale_experiment.py` from Drive
3. Patches it to add `TwoDALiBi` class and `alibi_2d` PE type
4. Trains 3 seeds × 300 epochs each on ImageNet-100

**Output**: checkpoints saved to
`/content/drive/MyDrive/Trained models_ImageNet100/alibi_2d_seed{seed}/best_model.pth`

**Compute**: ~8.2h on NVIDIA RTX PRO 6000 Blackwell Server Edition per seed.

⚠️ Colab sessions disconnect after 24h. Train ONE seed at a time and
resume the next in a fresh session.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
print("Drive mounted.")


## 2. Configuration

Edit the paths if your Drive layout differs. The defaults match the
existing `Trained models_ImageNet100` folder structure.


In [ ]:
# ============================================================
# CONFIG — edit if your paths differ
# ============================================================

# Where your existing full_scale_experiment.py lives
ORIG_SCRIPT_PATH = "/content/drive/MyDrive/full_scale_experiment.py"

# Where the 2D-ALiBi patch script lives (upload it to Drive first)
PATCH_SCRIPT_PATH = "/content/drive/MyDrive/apply_2d_alibi_patch.py"

# Working directory in the Colab session (not Drive)
WORK_DIR = "/content/work"

# Patched script will be written here
PATCHED_SCRIPT = os.path.join(WORK_DIR, "full_scale_experiment_v2.py")

# ImageNet-100 data
DATA_DIR = "/content/drive/MyDrive/imagenet100_resized"

# Output directory for checkpoints — should match where the rest of
# your IN-100 models live, so the analysis notebook finds them
OUTPUT_DIR = "/content/drive/MyDrive/Trained models_ImageNet100"

# Which seed to train in THIS session
# Change between sessions: 42 -> 123 -> 456
CURRENT_SEED = 42

# Training settings
EPOCHS      = 300
BATCH_SIZE  = 128
NUM_WORKERS = 4

# Quick pilot to verify everything works before committing 25h
RUN_PILOT_FIRST = True   # set False once you've verified
PILOT_EPOCHS    = 5

print(f"Original script:  {ORIG_SCRIPT_PATH}")
print(f"Patch script:     {PATCH_SCRIPT_PATH}")
print(f"Patched output:   {PATCHED_SCRIPT}")
print(f"Data dir:         {DATA_DIR}")
print(f"Output dir:       {OUTPUT_DIR}")
print(f"Current seed:     {CURRENT_SEED}")
print(f"Epochs:           {EPOCHS}")
print(f"Run pilot first?  {RUN_PILOT_FIRST} ({PILOT_EPOCHS} epochs)")


## 3. Verify GPU available

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU! Switch runtime: Runtime > Change runtime type > GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")


## 4. Install dependencies

In [ ]:
!pip install -q timm tqdm scikit-learn scipy matplotlib
print("Dependencies installed.")


## 5. Verify required files are in Drive

Before running this cell, make sure you have uploaded TWO files to your Drive:

1. **`full_scale_experiment.py`** — your existing training script
   (the one that already works for the 4 PE methods)
2. **`apply_2d_alibi_patch.py`** — the patcher I gave you

Recommended location: `/content/drive/MyDrive/` (root of your Drive).


In [ ]:
import os

missing = []
for path, label in [(ORIG_SCRIPT_PATH, "original script"),
                     (PATCH_SCRIPT_PATH, "patch script"),
                     (DATA_DIR, "data directory")]:
    if not os.path.exists(path):
        missing.append(f"  ✗ {label}: {path}")
    else:
        print(f"  ✓ Found {label}: {path}")

if missing:
    print()
    print("MISSING FILES:")
    print("\n".join(missing))
    raise FileNotFoundError("Upload the missing files to Drive before continuing.")
else:
    print()
    print("All required files present.")


## 6. Apply the 2D-ALiBi patch

This produces a patched version of `full_scale_experiment.py` in the
Colab working directory. The original on Drive is NOT modified.


In [ ]:
import os
import shutil

os.makedirs(WORK_DIR, exist_ok=True)

# Copy patch script into work dir so we can run it locally
local_patch = os.path.join(WORK_DIR, "apply_2d_alibi_patch.py")
shutil.copy(PATCH_SCRIPT_PATH, local_patch)
print(f"Patch script copied to: {local_patch}")

# Apply the patch
cmd = f"cd {WORK_DIR} && python apply_2d_alibi_patch.py '{ORIG_SCRIPT_PATH}' '{PATCHED_SCRIPT}'"
print(f"Running: {cmd}\n")
exit_code = os.system(cmd)

if exit_code != 0:
    raise RuntimeError(f"Patch failed with exit code {exit_code}")

print(f"\nPatch applied successfully.")
print(f"Patched script size: {os.path.getsize(PATCHED_SCRIPT):,} bytes")
print(f"Original size:       {os.path.getsize(ORIG_SCRIPT_PATH):,} bytes")


## 7. Pilot run (recommended for first time)

Runs just 5 epochs to verify:
- Patched script imports cleanly
- 2D-ALiBi model builds and forward-passes
- Data loader works
- Loss decreases epoch-over-epoch

If the pilot completes without errors and loss decreases, you can
confidently commit to the full 300-epoch run. Pilot takes ~15-20 min.


In [ ]:
if RUN_PILOT_FIRST:
    pilot_output = "/content/pilot_test_2d_alibi"
    os.makedirs(pilot_output, exist_ok=True)

    cmd = f'''python {PATCHED_SCRIPT} \
        --data_dir "{DATA_DIR}" \
        --output_dir "{pilot_output}" \
        --pe_type alibi_2d \
        --seed 42 \
        --mode train \
        --epochs {PILOT_EPOCHS} \
        --batch_size {BATCH_SIZE} \
        --num_workers {NUM_WORKERS}'''

    print("PILOT RUN — 5 epochs, ~15-20 minutes")
    print("=" * 60)
    print(cmd)
    print("=" * 60)
    print()

    exit_code = os.system(cmd)

    if exit_code != 0:
        raise RuntimeError(f"Pilot failed with exit code {exit_code}")

    print()
    print("✅ Pilot complete. Inspect loss curve above.")
    print("If loss decreased over epochs, proceed to the full run.")
    print()
    print("To skip pilot in future sessions, set RUN_PILOT_FIRST = False in config cell.")
else:
    print("Pilot skipped (RUN_PILOT_FIRST = False).")


## 8. Full training run

Trains a single seed for 300 epochs. Expected wall time: ~25-30h H100.

⚠️ **Colab disconnects after ~24h**. Strategy:
- Train ONE seed per Colab session
- Best model checkpoint is saved to Drive every epoch (when val_acc improves)
- If session disconnects mid-run, the best checkpoint up to that point
  is preserved
- For Colab Pro+: keep the tab visible to avoid idle timeout

After this cell completes, change `CURRENT_SEED` to the next value
(42 → 123 → 456) and restart the runtime / re-run all cells.


In [ ]:
cmd = f'''python {PATCHED_SCRIPT} \
    --data_dir "{DATA_DIR}" \
    --output_dir "{OUTPUT_DIR}" \
    --pe_type alibi_2d \
    --seed {CURRENT_SEED} \
    --mode train \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --num_workers {NUM_WORKERS}'''

print(f"FULL TRAINING — seed {CURRENT_SEED}, {EPOCHS} epochs")
print("=" * 60)
print(cmd)
print("=" * 60)
print()
print(f"Expected wall time: ~25-30h H100")
print(f"Best checkpoint will be saved to:")
print(f"  {OUTPUT_DIR}/alibi_2d_seed{CURRENT_SEED}/best_model.pth")
print()

exit_code = os.system(cmd)

if exit_code != 0:
    print(f"\n⚠️ Training exited with code {exit_code}")
    print("Check the output above. If Colab disconnected, the latest best")
    print("checkpoint should still be in Drive.")
else:
    print(f"\n✅ Training complete for seed {CURRENT_SEED}.")
    print(f"\nNext step: change CURRENT_SEED to the next value")
    print(f"(42 -> 123 -> 456) and restart the runtime.")


## 9. Status: which seeds are done

In [ ]:
import os
print("2D-ALiBi training status:")
print("=" * 50)
for seed in [42, 123, 456]:
    ckpt = os.path.join(OUTPUT_DIR, f"alibi_2d_seed{seed}", "best_model.pth")
    if os.path.isfile(ckpt):
        size_mb = os.path.getsize(ckpt) / 1e6
        print(f"  ✓ seed {seed:3d}: {ckpt} ({size_mb:.0f} MB)")
    else:
        print(f"  ✗ seed {seed:3d}: not yet trained")
print()
print("When all 3 are done, run revision_analysis_v2.ipynb with")
print("  PE_TYPES = ['learned', 'sinusoidal', 'rope', 'alibi', 'alibi_2d']")
print("to compute MI, intrinsic analyses, and probes on the new models.")
